# 实验二 · 梯形积分法 —— 数据环境、同步与归约

**所属**：《并行计算技术》第五章 · OpenMP 编程　|　**难度**：⭐⭐ 基础　|　**预计时长**：30–40 分钟

本实验为第五章的核心案例。它以数值积分为载体，系统展示由「手工分解 + 加锁」到「工作分担 + 归约」的演进过程，并在此过程中建立本章最重要的一组概念：变量的共享与私有、临界区、归约子句。

> **实验说明**
> 1. 本实验采用**递进式的版本组织**：以串行实现为基准，每个版本仅引入一种新的 OpenMP 构造或一处相应的代码改写，并在同一次运行中完成全部版本的计时与正确性校验，因而各版本面对的是完全相同的数据与运行环境，各版本之间具有可比性。
> 2. 请自上而下依次执行各单元格（Shift+Enter）。
> 3. 本实验依赖支持 OpenMP 的 **GCC 编译器**，建议在华为鲲鹏处理器或其他 AArch64 平台上运行。
> 4. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。
> 5. 本实验的四个版本在容差范围内给出**一致的计算结果**（浮点加法次序不同，末位可能存在微小差异），差别仅在于同步方式与代码形态。重点关注耗时的变化，以及并行相关代码由约二十行缩减为三行的过程。
> 6. 命令行参数 `n` 必须能被线程数整除。V1 与 V2 采用手工块分解，该约束是手工分解方式的固有局限之一。

## 🎯 学习目标

完成本实验后，学生应能够：

- 复述梯形积分法的离散公式，并说明其为何属于**可归约**的计算模式
- 掌握 OpenMP 数据环境的**默认规则**：并行区域外声明的变量默认共享，区域内声明的变量为私有
- 正确使用 `private`、`firstprivate`、`shared`、`default(none)` 四个子句，并说明 `private` **不复制初值、不回写**这一性质
- 理解 `#pragma omp critical` 的作用与开销，说明为何它无法解决全部同步问题
- 掌握 `reduction` 子句的三步机制：创建私有副本、以单位元初始化、区域结束时合并
- 区分 `#pragma omp parallel` 与 `#pragma omp for`，理解工作分担构造如何自动划分迭代空间
- 解释浮点归约结果为何与串行结果存在微小差异，并给出合理的校验容差

## 🗺️ 学习路径

1. **准备阶段**：理解梯形积分的离散公式与手工块分解方案
2. **V0 · 串行基准**：确立正确性参照与性能基准
3. **V1 · 手工分解 + `critical`**：各线程分别计算局部结果，再依次进入临界区累加
4. **V2 · 手工分解 + `reduction`**：保留手工分解，把临界区替换为归约子句
5. **V3 · `parallel for` + `reduction`**：将迭代空间的划分交由编译器与运行时完成，代码缩减为三行
6. **可视化与分析**：绘制加速比柱状图，讨论临界区开销与代码简洁性的关系
7. **扩展实验**：线程数扫描，观察扩展性曲线

## 1. 背景知识：梯形积分法

### 1.1 数学原理

求定积分 $\int_a^b f(x)\,dx$ 数值解的一种基本方法，是将积分区间 $[a,b]$ 等分为 $n$ 个小区间，以梯形面积近似各小区间上曲边梯形的面积，再求和。

设步长 $h = (b-a)/n$，第 $i$ 个梯形的面积为 $\frac{h}{2}\left[f(x_i) + f(x_{i+1})\right]$。把 $n$ 个梯形的面积相加，各内部节点的函数值被相邻的两个梯形分别计入一次，整理后得到：

$$\int_a^b f(x)\,dx \approx h\left[\frac{f(a)+f(b)}{2} + \sum_{i=1}^{n-1} f(a+ih)\right]$$

本实验取 $f(x) = \dfrac{4}{1+x^2}$，积分区间为 $[0, 1]$。该积分的精确值为 $\pi$，因此计算结果可与已知常数对照，便于判断算法是否正确。

### 1.2 案例的教学价值

梯形积分法具有以下三项特点，适合作为本章的并行化教学案例：

- **计算模式典型**。它是一个标准的**归约**：大量独立的局部计算，最终归并为一个标量。科学计算中的求和、求最值、求内积都属于这一模式。
- **各次求和彼此独立**。第 $i$ 项与第 $j$ 项的计算之间没有任何依赖，因而不存在后面实验将要讨论的循环携带依赖问题，可以专注于数据环境本身。
- **正确性易于判断**。结果应当接近 $\pi$，任何明显的偏差都表明实现存在缺陷。

### 1.3 计算量与访存特征

该循环每次迭代约执行 2 次浮点乘法、3 次浮点加法与 1 次浮点除法（其中除法的延迟通常远高于加法与乘法，是本循环的主要开销），而其工作集仅为若干个标量，可完全驻留于寄存器，几乎不产生访存流量。因此该负载属于**计算受限**（compute-bound）类型，其加速比基本不受访存带宽制约，便于观察同步机制本身对性能的影响。这一点与实验七的矩阵向量乘法形成对照，后者是典型的访存受限负载。

## 2. 并行化方案：手工块分解

### 2.1 分解思路

把 $n$ 个小区间平均分给 `thread_count` 个线程，每个线程负责连续的 `local_n = n / thread_count` 个区间：

```text
  区间 [a, b] 被 n 等分，再按线程数分块：

  a                                               b
  ├───────────┼───────────┼───────────┼───────────┤
  │  线程 0   │  线程 1    │  线程 2   │  线程 3   │
  └───────────┴───────────┴───────────┴───────────┘
      local_a ↑           ↑ local_b
```

线程 `r` 负责的子区间为：

$$\texttt{local\_a} = a + r \cdot \texttt{local\_n} \cdot h, \qquad \texttt{local\_b} = \texttt{local\_a} + \texttt{local\_n} \cdot h$$

每个线程在各自的子区间上独立应用梯形公式，得到局部结果 `my_result`，最后将全部局部结果相加。

### 2.2 分解方案的正确性验证

各线程分别对各自的子区间应用梯形公式时，子区间的两个端点均按「半权」计入（即 $\frac{f(\text{local\_a})+f(\text{local\_b})}{2}$）。此处的「半权」与下文的「整权」，指该节点的函数值在求和式中的系数为 $1/2$ 或 $1$。相邻两个子区间的公共分界点，会被左侧线程当作 `local_b` 计入一次半权，又被右侧线程当作 `local_a` 计入一次半权，合计恰好一个整权。而全局的两个端点 $a$ 与 $b$ 各只被计入一次半权。

这与串行公式完全吻合，因此**手工分解与串行公式在数学上完全等价，不引入任何额外误差**；两者的差异仅来自浮点加法次序的不同。

### 2.3 分解方案带来的约束

上述分解要求 `n` 能被 `thread_count` 整除，否则末尾的若干区间将被遗漏。源码中对此做了参数合法性检查并给出明确提示。这一约束是**手工分解特有**的：V3 改用 `#pragma omp for` 之后，迭代空间的划分由运行时负责，余数会被自动分配，约束随之消失。这正是工作分担构造的价值之一。

## 3. OpenMP 关键知识点

### 3.1 数据环境的默认规则

进入并行区域后，每个变量或者被全体线程共用（**共享**），或者由每个线程各自持有一份独立副本（**私有**）。OpenMP 的默认数据属性由**变量的声明位置**决定。

| 变量的声明位置 | 默认属性 | 存放位置 |
|---|---|---|
| 并行区域**之外**声明的局部变量 | 共享 | 遇到该并行区域的线程（通常为主线程）的栈 |
| 并行区域**之内**声明的变量 | 私有 | 各线程自己的栈 |
| 被 `#pragma omp for` 分担的循环变量 | 私有（自动） | 各线程自己的栈 |
| 全局变量、`static` 变量 | 共享 | 静态存储区 |
| 由 `malloc` 分配的内存 | 共享（指针本身可私有） | 堆 |

> **理解要点**：共享与私有的区分，本质上是**变量存放在哪块内存**的区分。各线程拥有各自独立的栈，因此在并行区域内声明的变量互不干扰；而堆与静态存储区为全体线程共用，其中的数据默认即是共享的。

```text
   ┌──────────── 进程地址空间 ──────────────┐
   │  线程0栈   线程1栈   线程2栈  ...      │  ← 私有
   ├───────────────────────────────────────┤
   │            堆（malloc）                │  ← 共享
   ├───────────────────────────────────────┤
   │        全局变量 / static 变量          │  ← 共享
   └───────────────────────────────────────┘
```

### 3.2 四个数据环境子句

| 子句 | 语义 | 进入区域时 | 离开区域时 |
|---|---|---|---|
| `shared(x)` | 全体线程共用同一个 `x` | 不变 | 不变 |
| `private(x)` | 每线程一份副本 | **值未定义** | **不回写** |
| `firstprivate(x)` | 每线程一份副本 | 复制原值 | 不回写 |
| `lastprivate(x)` | 每线程一份副本 | 值未定义 | 回写**最后一次迭代**的值 |

**使用 `private` 时需特别注意以下两点**：

1. **不复制初值**。`double factor = 1.0;` 加上 `private(factor)` 之后，各线程副本的初值是**未定义的**，而不是 1.0。若需要复制初值，应使用 `firstprivate`。
2. **不回写**。区域结束后，外层变量仍保持进入前的值，线程内的任何修改都不会保留。若需要保留最后一次迭代的结果，应使用 `lastprivate`。

### 3.3 `default(none)`：强制显式声明数据属性

```c
#pragma omp parallel for default(none) shared(a, n) private(i, tmp)
```

加上 `default(none)` 之后，编译器要求区域内使用的**每一个**变量都必须显式声明其属性，任何遗漏都会导致编译错误。虽然书写量有所增加，但它迫使程序员在编译阶段逐一确认每个变量的归属，从而把「变量属性未经确认」这一隐患暴露在**编译期**。

> **需要澄清的边界**：`default(none)` 检查的是变量属性是否被显式声明，编译器并不分析程序中是否真的存在数据竞争。被显式声明为 `shared` 的变量仍然可能被并发写入，因此「编译通过」不等于「没有竞争」。

在工程实践中，OpenMP 程序的常见缺陷之一是将本应私有的变量默认处理为共享。`default(none)` 是一种成本很低的防范手段，建议在正式代码中采用。

### 3.4 `critical`：临界区

```c
#pragma omp critical
*result += my_result;
```

`critical` 保证同一时刻至多一个线程执行其后的结构化块，其作用类似于第四章中的互斥锁（`pthread_mutex_lock` / `pthread_mutex_unlock`）。

需要理解两点：

- **临界区是全局的**。无名 `critical` 构造在整个程序中共用同一个互斥对象，即便两处临界区保护的是完全无关的数据，也会相互阻塞。可用 `#pragma omp critical(name)` 指定名称加以区分。
- **临界区内的执行是串行化的**。构造内的代码由各线程依次执行，因此临界区内应当只放置尽可能少的语句。本实验 V1 即体现了这一原则：先在临界区**之外**完成整个局部积分的计算，仅将一次累加操作置于临界区内。

### 3.5 `reduction`：归约子句

```c
#pragma omp parallel for reduction(+ : result)
```

归约子句在语义上完成「为每个线程创建私有副本、在私有副本上累加、并在区域结束时合并到原变量」这一过程，相应代码由编译器自动生成。其执行过程分为三步：

1. **创建私有副本**。为每个线程创建一份 `result` 的私有副本。
2. **以单位元初始化**。副本的初值由运算符决定，而**不是**原变量的值：

   | 运算符 | 初值 | 运算符 | 初值 |
   |---|---|---|---|
   | `+` `-` | 0 | `*` | 1 |
   | `&` | 全 1 | `\|` `^` | 0 |
   | `&&` | 1 | `\|\|` | 0 |
   | `max` | 该类型最小值 | `min` | 该类型最大值 |

   > `-` 归约在 OpenMP 5.0 中已被列为**弃用特性**（其合并语义与 `+` 相同），新编写的代码不应再使用。

3. **区域结束时合并**。把全部私有副本按该运算符累加到**原变量的原有取值**上。

> **第 3 步的含义**：归约的最终结果是「原值 ⊕ 各线程副本」，而非「各线程副本」。本实验 V3 中 `result` 在进入并行区域前已被赋值为 $\frac{f(a)+f(b)}{2}$，这一项因而被正确地保留在最终结果中。

**归约的实现方式**：OpenMP 规范并未规定私有副本的合并方式，具体做法由实现决定——可能是经 $\lceil\log_2 p\rceil$ 轮两两合并的**树形归约**，也可能是以原子操作完成的线性合并。无论采用哪一种，其开销都显著低于让 $p$ 个线程依次进入全局临界区：后者的代价不仅来自串行化，还来自互斥对象的获取、释放及其竞争。这是 V2 通常快于 V1 的主要原因。下图为树形合并的示意，实际实现可能与此不同。

```text
   线程:  0    1    2    3    4    5    6    7
          │    │    │    │    │    │    │    │
          └─┬──┘    └─┬──┘    └─┬──┘    └─┬──┘   第 1 轮：4 次加法
            └────┬────┘         └────┬────┘      第 2 轮：2 次加法
                 └────────┬──────────┘           第 3 轮：1 次加法
                       最终结果
```

### 3.6 `parallel for`：工作分担构造

实验一已经指出，`#pragma omp parallel` 只是使线程组中的各个线程执行同一个结构化块。要让各线程分担**不同**的迭代，需要工作分担构造（worksharing construct）`#pragma omp for`：

```c
#pragma omp parallel        // 创建线程组
{
#pragma omp for             // 划分紧随其后的循环
  for (i = 0; i < n; i++) { ... }
}
```

当并行区域内只有一个循环时，可合写为 `#pragma omp parallel for`。

`#pragma omp for` 对被分担的循环有形式要求，称为**规范形式**（canonical form）：

- 循环次数在进入循环前即可确定，因此不允许 `while` 形式，循环体内不得出现跳出该循环的控制转移语句（`break`、`goto`、`return`）；内层循环或 `switch` 中的 `break` 以及 `continue` 不受此限制；
- 循环变量必须是整型或指针，且**在循环体内不得被修改**；
- 循环的上下界与步长在循环执行期间保持不变。

满足规范形式只是编译器接受该循环的条件，**并不代表该循环可以被安全并行**——迭代之间是否独立，需要程序员自行保证。实验三将专门讨论这一点。

## 4. 环境准备与检查

本节确认三项内容：编译器是否支持 OpenMP、运行时报告的处理器数量、以及各处理器核心的最大频率是否一致。

第三项检查针对**异构多核**平台。Arm 的 big.LITTLE 架构把高性能核心与高能效核心集成在同一块芯片上，二者的频率与微架构均不相同。在这类平台上，同一段代码在不同类型的核心上执行，耗时可能相差 2 倍以上，线程数与加速比之间因而不再是简单的线性关系。

需要说明的是，最大频率不一致只是异构多核的**必要非充分**证据：同构多核平台也可能因加速频率（boost）策略或芯片分级（binning）而上报不同的 `cpuinfo_max_freq`。因此下面的检查只给出提示，确认平台是否为异构架构还需结合 `lscpu` 输出的核心型号信息。

In [ ]:
import os
import re
import subprocess
import platform

print('=' * 60)
print(' 一、平台信息')
print('=' * 60)
print('操作系统   :', platform.system(), platform.release())
print('处理器架构 :', platform.machine())
print('逻辑核心数 :', os.cpu_count())

print()
print('=' * 60)
print(' 二、编译器与 OpenMP 支持')
print('=' * 60)
gcc_ver = subprocess.run(['gcc', '--version'], capture_output=True,
                         text=True).stdout.splitlines()[0]
print('编译器     :', gcc_ver)

probe = subprocess.run('echo | gcc -fopenmp -dM -E -x c - | grep _OPENMP',
                       shell=True, capture_output=True, text=True).stdout.strip()
if probe:
    ver = int(probe.split()[-1])
    spec = {200805: '3.0', 201107: '3.1', 201307: '4.0',
            201511: '4.5', 201811: '5.0', 202011: '5.1'}.get(ver, '未知')
    print('_OPENMP    :', ver, '(对应 OpenMP %s 规范)' % spec)
    print('数组段归约 :', '支持' if ver >= 201511 else '不支持（需要 4.5 及以上）')
else:
    print('⚠️  未检测到 OpenMP 支持，请确认编译时带有 -fopenmp')

print()
print('=' * 60)
print(' 三、核心频率与异构性检查')
print('=' * 60)
freqs = []
for cpu in range(os.cpu_count() or 1):
    path = '/sys/devices/system/cpu/cpu%d/cpufreq/cpuinfo_max_freq' % cpu
    try:
        with open(path) as f:
            freqs.append((cpu, int(f.read().strip()) // 1000))
    except OSError:
        pass

if not freqs:
    print('无法读取 cpufreq 节点，跳过异构性检查。')
else:
    for cpu, mhz in freqs:
        print('  CPU%-2d 最大频率: %5d MHz' % (cpu, mhz))
    distinct = sorted(set(m for _, m in freqs))
    if len(distinct) > 1:
        print()
        print('⚠️  检测到 %d 种不同的最大频率，本平台可能为异构多核架构（如 Arm big.LITTLE）。'
              % len(distinct))
        print('    请结合 lscpu 输出的核心型号信息进一步确认。')
        print('    若确为异构平台，测速前建议执行：')
        print('      export OMP_PROC_BIND=close')
        print('      export OMP_PLACES=cores')
    else:
        print()
        print('✅ 全部核心的最大频率一致，可按同构多核平台处理。')

print()
print('OMP_NUM_THREADS =', os.environ.get('OMP_NUM_THREADS', '（未设置，由运行时决定）'))
print('OMP_PROC_BIND   =', os.environ.get('OMP_PROC_BIND', '（未设置）'))
print('OMP_PLACES      =', os.environ.get('OMP_PLACES', '（未设置）'))

## 5. 实验工具函数

本节定义三个贯穿全章的辅助函数，后续各实验均直接调用，不再重复说明。

| 函数 | 作用 |
|---|---|
| `compile_c(src)` | 以 `-O3 -fopenmp -Wall -Wextra` 编译指定源文件，并回显全部告警 |
| `run_c(binary, *args)` | 运行可执行文件并原样打印其标准输出 |
| `parse_table(output)` | 从程序输出的结果表中提取「方法名 / 耗时 / 加速比 / 校验」四列 |

**关于编译选项**：全章统一使用 `-O3 -fopenmp`。AArch64 平台的 NEON 属于基线指令集，无需附加 `-march` 或 `-mcpu` 选项。`-Wall -Wextra` 用于暴露数据环境声明不当引发的告警，这类告警在 OpenMP 程序中往往是并发缺陷的征兆，不应忽略。

In [ ]:
import subprocess
import re
import os

SRC_DIR = 'src_trapezoid'
os.makedirs(SRC_DIR, exist_ok=True)

CFLAGS = ['-O3', '-fopenmp', '-Wall', '-Wextra']


def compile_c(src, extra=('-lm',)):
    """编译单个源文件，返回可执行文件路径；编译失败时抛出异常。"""
    src_path = os.path.join(SRC_DIR, src)
    binary = os.path.join(SRC_DIR, os.path.splitext(src)[0])
    cmd = ['gcc'] + CFLAGS + ['-o', binary, src_path] + list(extra)
    print('$', ' '.join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.stdout.strip():
        print(proc.stdout.rstrip())
    if proc.stderr.strip():
        print(proc.stderr.rstrip())
    if proc.returncode != 0:
        raise RuntimeError('编译失败：%s' % src)
    print('✅ 编译通过，无告警' if not proc.stderr.strip()
          else '⚠️  编译通过，但存在告警，请逐条阅读')
    return binary


def run_c(binary, *args, env=None):
    """运行可执行文件，打印并返回其标准输出。"""
    cmd = [binary] + [str(a) for a in args]
    print('$', ' '.join(cmd))
    print()
    run_env = dict(os.environ)
    if env:
        run_env.update({k: str(v) for k, v in env.items()})
    proc = subprocess.run(cmd, capture_output=True, text=True, env=run_env)
    print(proc.stdout.rstrip())
    if proc.stderr.strip():
        print('[stderr]', proc.stderr.rstrip())
    return proc.stdout


ROW_RE = re.compile(r'^\|\s*(.+?)\s*\|\s*([0-9.]+)\s*\|\s*([0-9.]+)x\s*\|\s*(\S+)\s*\|$')


def parse_table(output):
    """解析结果表，返回 [(方法名, 耗时ms, 加速比, 校验结论), ...]。"""
    rows = []
    for line in output.splitlines():
        m = ROW_RE.match(line.strip())
        if m:
            rows.append((m.group(1), float(m.group(2)),
                         float(m.group(3)), m.group(4)))
    return rows


print('工具函数已就绪，源码目录：', os.path.abspath(SRC_DIR))

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

C_BASE, C_GOOD, C_FAIL, C_SLOW = '#7f7f7f', '#1f77b4', '#d62728', '#ff7f0e'


def plot_speedup(rows, title, figsize=(10, 5)):
    """绘制加速比柱状图。配色：灰=基准，蓝=有效加速，橙=慢于基准，红=校验失败。"""
    if not rows:
        print('未解析到结果行，请先运行上一单元格。')
        return
    names = [r[0] for r in rows]
    speeds = [r[2] for r in rows]
    colors = []
    for i, (_, _, sp, chk) in enumerate(rows):
        if i == 0:
            colors.append(C_BASE)
        elif chk == 'FAIL':
            colors.append(C_FAIL)
        elif sp < 1.0:
            colors.append(C_SLOW)
        else:
            colors.append(C_GOOD)

    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.bar(range(len(names)), speeds, color=colors,
                  edgecolor='black', linewidth=0.6, width=0.6)
    ax.axhline(1.0, color='black', linestyle='--', linewidth=1.0, alpha=0.7)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=20, ha='right', fontsize=9)
    ax.set_ylabel('Speedup vs. Serial Baseline')
    ax.set_title(title, fontsize=12, pad=12)
    ax.grid(axis='y', linestyle=':', alpha=0.5)
    ax.set_axisbelow(True)

    for bar, (_, ms, sp, chk) in zip(bars, rows):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.02,
                '%.2fx\n%.1f ms%s' % (sp, ms, '' if chk in ('-', 'PASS') else '\nFAIL'),
                ha='center', va='bottom', fontsize=8)

    ax.set_ylim(0, max(speeds) * 1.30)
    plt.tight_layout()
    plt.show()


print('绘图函数已就绪。配色：灰=基准，蓝=有效加速，橙=慢于基准，红=校验失败。')

## 6. 版本设计总览

| 版本 | 新增计算函数 | 分解方式 | 同步方式 | 结果表行号 |
|---|---|---|---|---|
| **V0** | `trap_serial` | — | — | 1 |
| **V1** | `trap_manual_critical` | 手工块分解 | `#pragma omp critical` | 2 |
| **V2** | `trap_manual_reduction` | 手工块分解 | `reduction(+ : ...)` | 3 |
| **V3** | `trap_parallel_for` | `#pragma omp for` 自动 | `reduction(+ : ...)` | 4 |

四个版本沿两条线索递进：

- **同步方式**：临界区 → 归约（V1 → V2）
- **分解方式**：手工计算区间 → 交由运行时划分（V2 → V3）

每一步只改动一处，因此耗时的变化可明确归因于该处改动。

### 6.1 代码规模的变化

| 版本 | 并行相关代码行数 | 说明 |
|---|---|---|
| V1 | 约 20 行 | 区间计算 6 行 + 局部积分 5 行 + 临界区 2 行 |
| V2 | 约 18 行 | 去掉临界区，改由子句表达 |
| V3 | **3 行** | 一行制导语句 + 一个循环 |

V3 与串行版 V0 的差别仅在于增加了一行 `#pragma` 制导语句（变量命名不计）。这正是编译制导相较显式线程管理最直观的优势。

## 7. 逐版本代码讲解

### 7.1 V0 · 串行基准

```c
static double trap_serial(double a, double b, long n) {
  double h = (b - a) / (double)n;
  double sum = (f(a) + f(b)) / 2.0;   // 两个端点各计半权

  for (long i = 1; i <= n - 1; i++) {
    sum += f(a + (double)i * h);      // 内部节点计整权
  }
  return sum * h;
}
```

循环从 `i = 1` 起、至 `i = n-1` 止，恰好覆盖全部内部节点。两个端点已在循环外以半权计入。

此版本同时作为**性能基准**与**正确性参照**。后续三个版本的校验结果，均以本版本的返回值为准。

### 7.2 V1 · 手工分解 + 临界区

```c
static void trap_manual_critical(double a, double b, long n,
                                 double *result) {
  int thread_count = omp_get_num_threads();
  int my_rank = omp_get_thread_num();

  double h = (b - a) / (double)n;
  long local_n = n / thread_count;
  double local_a = a + (double)my_rank * (double)local_n * h;
  double local_b = local_a + (double)local_n * h;

  double my_result = (f(local_a) + f(local_b)) / 2.0;
  for (long i = 1; i <= local_n - 1; i++) {
    my_result += f(local_a + (double)i * h);
  }
  my_result *= h;

#pragma omp critical
  *result += my_result;
}
```

调用形式为：

```c
res_v1 = 0.0;
#pragma omp parallel num_threads(thread_count)
  trap_manual_critical(a, b, n, &res_v1);
```

**设计要点如下**：

1. **函数内的局部变量自然具有私有属性**。`local_a`、`local_b`、`my_result` 都声明在函数体内，位于各线程各自的栈上，因此无需任何子句即可保证互不干扰。这是最为简洁、也最不易出错的私有化方式。
2. **临界区仅包含一条语句**。整个局部积分（可能是数百万次浮点运算）在临界区**之外**完成，进入临界区的只有一次累加。若将整个循环置于临界区内，程序将退化为串行执行，且因附加了互斥开销而比 V0 更慢。
3. **`*result` 必须在并行区域外清零**。此处的累加是**就地**（in-place）进行的：`*result` 在并行区域内被反复读改写，因此每次测量前需把 `res_v1` 重置为 0，且该重置动作应排除在计时窗口之外。

### 7.3 V2 · 手工分解 + 归约

V2 与 V1 的**唯一**区别是：删去临界区，改为返回局部结果，由归约子句合并。

```c
static double trap_manual_reduction(double a, double b, long n) {
  // ... 区间计算与局部积分部分与 V1 完全相同 ...
  return my_result * h;      // 不再写共享变量，直接返回
}
```

调用形式为：

```c
res_v2 = 0.0;
#pragma omp parallel num_threads(thread_count) reduction(+ : res_v2)
{ res_v2 += trap_manual_reduction(a, b, n); }
```

> **注意**：`reduction` 子句加在 `parallel` 构造上，而非 `for` 上。此处并没有需要分担的循环，归约的对象是「每个线程执行一次的表达式」。这说明 `reduction` 并不依附于 `for`，它是一个独立的数据环境子句。

V1 与 V2 的计算量完全相同，耗时差异**主要来自合并方式**：V1 是 $p$ 个线程依次串行进入同一个全局临界区，V2 则由 OpenMP 实现选择开销更低的合并方式（树形合并或原子合并，参见 3.5 节）。线程数越多，这一差距越明显。

### 7.4 V3 · 工作分担 + 归约

```c
static double trap_parallel_for(double a, double b, long n,
                                int thread_count) {
  double h = (b - a) / (double)n;
  double result = (f(a) + f(b)) / 2.0;

#pragma omp parallel for num_threads(thread_count) reduction(+ : result)
  for (long i = 1; i <= n - 1; i++) {
    result += f(a + (double)i * h);
  }
  return result * h;
}
```

将 V3 与 V0 的串行版本对照可见，**除增加一行 `#pragma` 制导语句外，两者的代码结构完全一致**（变量命名不计）。所有的区间计算、线程编号获取与局部结果合并，均由编译器生成的代码借助 OpenMP 运行时库在运行时完成。

**三处自动化**：

| 事项 | V1/V2 的做法 | V3 的做法 |
|---|---|---|
| 迭代空间划分 | 手工计算 `local_a`、`local_n` | 运行时自动划分 |
| 循环变量 `i` | 声明于函数内，自然具有私有属性 | 由构造自动私有化 |
| 余数处理 | 需要 `n % thread_count == 0` | 自动分配，无此约束 |

另外注意 `result` 的初值 $\frac{f(a)+f(b)}{2}$ 在归约后被正确保留，这正是 3.5 节第 3 步所述「合并到原变量的原有取值上」的体现。

## 8. 源代码写入

下面的单元格写入完整源码`omp_trapezoidal_rule.c`，阅读时请留意计时方法：每个版本均在 `NTIMES` 次循环内**逐次累加耗时后取平均**，而非取最优值。取平均可反映调度抖动对整体性能的影响；取最优值则更接近无干扰条件下的性能上界。本实验关注同步开销的实际影响，故采用前者。此外，重置累加器的语句一律排除在计时窗口之外。

In [ ]:
%%writefile {SRC_DIR}/omp_trapezoidal_rule.c
#define _POSIX_C_SOURCE 200809L

#include <math.h>
#include <omp.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#ifndef _OPENMP
#error "OpenMP is required. Please compile with -fopenmp."
#endif

#define NTIMES 20
#define MAX_THREADS 16
// The four versions differ only in the order of the floating point additions,
// so the absolute difference stays far below this bound.
#define TOL 1e-6

#define BANNER "============================================================"
#define LINE "------------------------------------------------------------"

// ----------------------------------------------------------------------------
// Common helpers
// ----------------------------------------------------------------------------
static double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

static int check_diff(const double *ref, const double *test, long n,
                      double tol) {
  for (long i = 0; i < n; i++) {
    if (fabs(ref[i] - test[i]) > tol) {
      return 0;
    }
  }
  return 1;
}

static void print_table_header(void) {
  printf("\n%s\n", LINE);
  printf("| %-26s | %9s | %7s | %-5s |\n", "Method", "Time(ms)", "Speedup",
         "Check");
  printf("|----------------------------|-----------|---------|-------|\n");
}

// check < 0 marks the baseline row, which has no correctness reference.
static void print_row(const char *name, double time_ms, double base_ms,
                      int check) {
  const char *status = (check < 0) ? "-" : (check ? "PASS" : "FAIL");
  double speedup = (time_ms > 0.0) ? base_ms / time_ms : 0.0;
  printf("| %-26s | %9.3f | %6.2fx | %-5s |\n", name, time_ms, speedup, status);
}

// ----------------------------------------------------------------------------
// The function to integrate. The integral of f over [0, 1] equals pi.
// ----------------------------------------------------------------------------
static double f(double x) { return 4.0 / (1.0 + x * x); }

// ============================================================================
// V0: Serial baseline
// ============================================================================
static double trap_serial(double a, double b, long n) {
  double h = (b - a) / (double)n;
  double sum = (f(a) + f(b)) / 2.0;

  for (long i = 1; i <= n - 1; i++) {
    sum += f(a + (double)i * h);
  }
  return sum * h;
}

// ============================================================================
// V1: Manual block decomposition + critical section
// Must be called from inside a #pragma omp parallel region.
// ============================================================================
static void trap_manual_critical(double a, double b, long n, double *result) {
  int thread_count = omp_get_num_threads();
  int my_rank = omp_get_thread_num();

  double h = (b - a) / (double)n;
  long local_n = n / thread_count;
  double local_a = a + (double)my_rank * (double)local_n * h;
  double local_b = local_a + (double)local_n * h;

  // local_a, local_b and my_result live on the private stack of each thread.
  double my_result = (f(local_a) + f(local_b)) / 2.0;
  for (long i = 1; i <= local_n - 1; i++) {
    my_result += f(local_a + (double)i * h);
  }
  my_result *= h;

  // *result is shared, so the read-modify-write must be serialised.
#pragma omp critical
  *result += my_result;
}

// ============================================================================
// V2: Manual block decomposition + reduction clause
// Must be called from inside a parallel region carrying reduction(+ : ...).
// ============================================================================
static double trap_manual_reduction(double a, double b, long n) {
  int thread_count = omp_get_num_threads();
  int my_rank = omp_get_thread_num();

  double h = (b - a) / (double)n;
  long local_n = n / thread_count;
  double local_a = a + (double)my_rank * (double)local_n * h;
  double local_b = local_a + (double)local_n * h;

  double my_result = (f(local_a) + f(local_b)) / 2.0;
  for (long i = 1; i <= local_n - 1; i++) {
    my_result += f(local_a + (double)i * h);
  }
  return my_result * h;
}

// ============================================================================
// V3: Worksharing construct. The runtime splits the iteration space and the
// reduction clause creates one private copy of result per thread.
// ============================================================================
static double trap_parallel_for(double a, double b, long n, int thread_count) {
  double h = (b - a) / (double)n;
  double result = (f(a) + f(b)) / 2.0;

#pragma omp parallel for num_threads(thread_count) reduction(+ : result)
  for (long i = 1; i <= n - 1; i++) {
    result += f(a + (double)i * h);
  }
  return result * h;
}

// ----------------------------------------------------------------------------
// Driver
// ----------------------------------------------------------------------------
int main(int argc, char *argv[]) {
  if (argc != 5) {
    printf("Usage: %s <a> <b> <n> <thread_count>\n", argv[0]);
    printf("Example: %s 0.0 1.0 10000000 4\n", argv[0]);
    return 1;
  }

  double a = strtod(argv[1], NULL);
  double b = strtod(argv[2], NULL);
  long n = strtol(argv[3], NULL, 10);
  int thread_count = (int)strtol(argv[4], NULL, 10);

  if (n <= 0) {
    printf("Error: n must be > 0\n");
    return 1;
  }
  if (thread_count < 1 || thread_count > MAX_THREADS) {
    printf("Error: thread_count must be between 1 and %d\n", MAX_THREADS);
    return 1;
  }
  if (n % thread_count != 0) {
    printf("Error: n must be divisible by thread_count.\n");
    printf("       V1 and V2 use block decomposition and would otherwise\n");
    printf("       drop the trailing sub-intervals.\n");
    return 1;
  }

  printf("%s\n", BANNER);
  printf(" Lab 2: Trapezoidal Rule\n");
  printf(" Interval: [%.2f, %.2f] | n: %ld | Threads: %d | Runs: %d\n", a, b, n,
         thread_count, NTIMES);
  printf(" _OPENMP: %d | Procs: %d\n", _OPENMP, omp_get_num_procs());
  printf("%s\n", BANNER);

  double start = 0.0;
  double t_v0 = 0.0, t_v1 = 0.0, t_v2 = 0.0, t_v3 = 0.0;
  double res_v0 = 0.0, res_v1 = 0.0, res_v2 = 0.0, res_v3 = 0.0;

  // --- V0: serial baseline ---
  for (int t = 0; t < NTIMES; t++) {
    start = get_time_ms();
    res_v0 = trap_serial(a, b, n);
    t_v0 += get_time_ms() - start;
  }
  t_v0 /= NTIMES;

  // --- V1: manual decomposition + critical ---
  for (int t = 0; t < NTIMES; t++) {
    res_v1 = 0.0;
    start = get_time_ms();
#pragma omp parallel num_threads(thread_count)
    trap_manual_critical(a, b, n, &res_v1);
    t_v1 += get_time_ms() - start;
  }
  t_v1 /= NTIMES;

  // --- V2: manual decomposition + reduction ---
  for (int t = 0; t < NTIMES; t++) {
    res_v2 = 0.0;
    start = get_time_ms();
#pragma omp parallel num_threads(thread_count) reduction(+ : res_v2)
    { res_v2 += trap_manual_reduction(a, b, n); }
    t_v2 += get_time_ms() - start;
  }
  t_v2 /= NTIMES;

  // --- V3: parallel for + reduction ---
  for (int t = 0; t < NTIMES; t++) {
    start = get_time_ms();
    res_v3 = trap_parallel_for(a, b, n, thread_count);
    t_v3 += get_time_ms() - start;
  }
  t_v3 /= NTIMES;

  print_table_header();
  print_row("V0: Serial Baseline", t_v0, t_v0, -1);
  print_row("V1: Manual + Critical", t_v1, t_v0,
            check_diff(&res_v0, &res_v1, 1, TOL));
  print_row("V2: Manual + Reduction", t_v2, t_v0,
            check_diff(&res_v0, &res_v2, 1, TOL));
  print_row("V3: For + Reduction", t_v3, t_v0,
            check_diff(&res_v0, &res_v3, 1, TOL));
  printf("%s\n", LINE);

  printf("\nIntegral value (V0): %.15f\n", res_v0);

  return 0;
}

## 9. 编译与运行

### 9.1 编译

本程序使用了 `fabs`，需要链接数学库 `-lm`。

In [ ]:
bin_trap = compile_c('omp_trapezoidal_rule.c')

### 9.2 运行

参数依次为：积分下限、积分上限、区间数、线程数。

取 $n = 10^7$ 是为了使单次计算的耗时达到毫秒量级，从而让线程创建与同步的开销在总耗时中占比适中：既不至于被计算时间完全掩盖，也不至于主导总耗时。

In [ ]:
out_trap = run_c(bin_trap, 0.0, 1.0, 10000000, 4)
rows_trap = parse_table(out_trap)
print()
print('解析到 %d 行结果：' % len(rows_trap))
for name, ms, sp, chk in rows_trap:
    print('  %-28s %8.3f ms  %5.2fx  %s' % (name, ms, sp, chk))

## 10. 结果可视化

以上一单元格的运行输出绘制加速比柱状图。配色约定：灰色为串行基准，蓝色为取得有效加速的版本，橙色为慢于基准的版本，红色为校验未通过的版本。

In [ ]:
plot_speedup(rows_trap,
             'Lab 2: Trapezoidal Rule (n = 1e7, 4 threads)')

## 11. 结果分析

> 以下结论针对**趋势规律**。具体数值随平台、线程数与编译器版本而变化，请以自己实验平台上的实测结果为准。

**① 四个版本的校验全部通过，说明并行化没有改变计算语义**

这是讨论性能的前提。若某个版本校验未通过，则其耗时数据不具备参考价值：计算结果错误的程序，其性能数据没有讨论意义。

**② V1 与 V2 的差距来自合并方式，而非计算本身**

两者的局部积分代码逐字相同，计算量完全一致。V1 让 $p$ 个线程依次进入同一个全局临界区，合并阶段的耗时随线程数**线性增长**；V2 的合并由 OpenMP 实现完成，其开销随线程数的增长明显更为平缓（若实现采用树形合并，则为对数增长）。

在本实验的参数下（$n = 10^7$，4 线程），合并阶段只发生 4 次，占总耗时的比例很小，因此两者差距有限。若把线程数提高到 8 或 16，或把 $n$ 减小到 $10^5$，差距会明显放大。第 12 节的练习 2 正是为观察这一点而设计的。

**③ V3 与 V2 性能相近，但代码规模相差数倍**

`#pragma omp for` 所生成的划分逻辑与手工编写的块分解在效率上没有本质差别，因此两者耗时接近。V3 的价值不在性能，而在于：

- 代码由约 18 行缩减为 3 行，降低了引入错误的可能性；
- 不再要求 `n` 能被线程数整除；
- 划分策略可以通过 `schedule` 子句灵活调整，无需改动循环体（实验四）。

**④ 关于加速比达不到线程数的原因**

本实验为计算受限负载，理想加速比应接近线程数。实测值通常低于该上限，主要来自三方面：并行区域的创建与销毁开销、归约合并的开销、以及并行区域出口处隐式栅栏造成的等待。在异构多核平台上，还需叠加高能效核心与高性能核心之间的速度差异——由于隐式栅栏的存在，整个并行区域的耗时由**执行时间最长的线程**决定。

**⑤ 积分结果与 $\pi$ 的偏差**

程序末尾打印的积分值与 $\pi$ 在小数点后第 13 位附近开始出现差异，偏差量级约为 $10^{-13}$。构成这一偏差的是两类性质完全不同的误差，须加以区分：

- **截断误差**。复合梯形公式的误差为 $-\dfrac{(b-a)h^{2}}{12}f''(\xi)$。在 $n = 10^{7}$（即 $h = 10^{-7}$）时，该项的量级仅约为 $10^{-15}$。它由算法本身决定，属于数学性质，与是否并行无关。
- **舍入误差**。串行版本需要把 $10^{7}$ 个双精度数依次累加，每次累加都引入一次舍入，其误差随累加长度不断累积。在本实验的参数下，该项的量级约为 $10^{-13}$，比截断误差高出约两个数量级，是上述偏差的**主要来源**。

由此可以得到一个初看有些反直觉的结论：**在这类长串累加的计算中，并行版本的结果往往比串行版本更接近真值**。V1 与 V2 把累加拆成 $p$ 段，每段的累加长度降为原来的 $1/p$，部分和的舍入误差随之减小。在本实验的参数下（$n = 10^{7}$，4 线程），分块求和相对 $\pi$ 的偏差约为 $10^{-14}$，比串行版本小一个数量级（具体数值随平台与编译器而异）。

正因为并行版本与串行版本各自带有不同的舍入误差，本实验的校验以**串行结果**为参照、并设定 `1e-6` 的容差，其目的是确认并行化没有改变**计算语义**，而不是评价数值精度。

## 12. 🔧 动手练习

**练习 1**　把 `n` 依次改为 $10^5$、$10^6$、$10^7$、$10^8$，观察加速比随问题规模的变化。请解释：为何问题规模越小，加速比越低？

**练习 2**　固定 $n = 10^6$，把线程数依次改为 1、2、4、8，重点观察 V1（临界区）与 V2（归约）之间的差距如何随线程数变化。

**练习 3**　故意传入一个不能被线程数整除的 `n`（例如 `10000001` 与 4 线程），记录程序的提示信息。再思考：若去掉该检查，V1 与 V2 会给出怎样的错误结果？V3 是否也会受影响？

**练习 4**（进阶）　把 V1 中的临界区扩大至覆盖整个局部积分循环，重新编译运行，记录耗时的变化并解释原因。

**练习 5**（进阶）　尝试在 V3 的制导语句上加入 `default(none)`，补齐全部变量的属性声明，使其通过编译。该过程可用于检验对各变量数据属性的掌握程度。

### 12.1 练习 1 的参考实现：问题规模扫描

In [ ]:
import matplotlib.pyplot as plt

sizes = [100000, 1000000, 10000000, 100000000]
scan_n = {}
for n in sizes:
    out = subprocess.run([bin_trap, '0.0', '1.0', str(n), '4'],
                         capture_output=True, text=True).stdout
    scan_n[n] = parse_table(out)
    print('n = %-10d  ' % n, end='')
    print('  '.join('%s %.2fx' % (r[0].split(':')[0], r[2])
                    for r in scan_n[n][1:]))

fig, ax = plt.subplots(figsize=(9, 5))
for idx, label in enumerate(['V1: Manual + Critical',
                             'V2: Manual + Reduction',
                             'V3: For + Reduction']):
    ys = [scan_n[n][idx + 1][2] for n in sizes]
    ax.plot(sizes, ys, marker='o', label=label)
ax.set_xscale('log')
ax.axhline(1.0, color='black', linestyle='--', linewidth=1.0, alpha=0.7)
ax.set_xlabel('Problem size n (log scale)')
ax.set_ylabel('Speedup vs. Serial Baseline')
ax.set_title('Lab 2: Speedup vs. Problem Size (4 threads)')
ax.grid(linestyle=':', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()

### 12.2 练习 2 的参考实现：线程数扩展性扫描

取 $n = 10^6$（较小的规模会放大同步开销的占比），线程数取 1、2、4、8。注意 $10^6$ 可被这四个数整除。

In [ ]:
threads = [1, 2, 4, 8]
N_FIX = 1000000
scan_t = {}
for nt in threads:
    out = subprocess.run([bin_trap, '0.0', '1.0', str(N_FIX), str(nt)],
                         capture_output=True, text=True).stdout
    scan_t[nt] = parse_table(out)
    print('threads = %-3d ' % nt, end='')
    print('  '.join('%s %.2fx' % (r[0].split(':')[0], r[2])
                    for r in scan_t[nt][1:]))

fig, ax = plt.subplots(figsize=(9, 5))
for idx, label in enumerate(['V1: Manual + Critical',
                             'V2: Manual + Reduction',
                             'V3: For + Reduction']):
    ys = [scan_t[nt][idx + 1][2] for nt in threads]
    ax.plot(threads, ys, marker='o', label=label)
ax.plot(threads, threads, color='gray', linestyle=':',
        label='Ideal (linear)')
ax.set_xlabel('Thread count')
ax.set_ylabel('Speedup vs. Serial Baseline')
ax.set_title('Lab 2: Scalability (n = 1e6)')
ax.set_xticks(threads)
ax.grid(linestyle=':', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()

## 13. 🤔 思考题

**思考题 1**　V1 中若把 `my_result` 的声明移到函数**外部**成为全局变量，程序会出现什么问题？临界区能否解决它？请从「变量存放在哪块内存」出发作答。

**思考题 2**　`reduction(+ : result)` 中，各线程私有副本的初值是 0，而 `result` 在进入区域前已被赋值为 $\frac{f(a)+f(b)}{2}$。若归约的初值规则改为「复制原变量的值」，最终结果会偏差多少？请用 $p$ 个线程的情形给出表达式。

**思考题 3**　浮点加法不满足结合律，因此 V1、V2、V3 的结果与串行版本在最后若干位上并不完全相同。本实验的校验容差取 `1e-6`。若把容差改为 `1e-16`，程序会报告什么？在工程实践中，应当依据什么来确定一个合理的容差？

**思考题 4**　V1 的临界区中只有一条 `*result += my_result;`。若把它改为 `#pragma omp atomic`，性能会变化吗？两者的适用范围有何不同？（提示：`atomic` 只能保护单条读改写语句。）

**思考题 5**　若将 V3 写成下面的形式：

```c
#pragma omp parallel num_threads(thread_count) reduction(+ : result)
for (long i = 1; i <= n - 1; i++) {
  result += f(a + (double)i * h);
}
```

注意这里是 `parallel` 而非 `parallel for`。程序能否通过编译？运行结果是否正确？总计算量会变成串行版本的多少倍？

**思考题 6**（综合）　本实验的四个版本中，哪一个版本的代码最容易在维护过程中被改错？请指出具体的风险点，并说明 `default(none)` 能否防范该风险。

## 14. 📌 本实验小结

| 概念 | 要点 |
|---|---|
| 数据环境默认规则 | 区域外声明的默认共享，区域内声明的默认私有 |
| `private` | 不复制初值、不回写 |
| `firstprivate` | 复制初值、不回写 |
| `lastprivate` | 不复制初值、回写最后一次迭代的值 |
| `default(none)` | 强制显式声明数据属性，在编译期暴露未经确认的变量 |
| `critical` | 全局互斥，合并阶段的开销随线程数线性增长 |
| `reduction` | 私有副本 + 单位元初始化 + 由实现选择的合并方式，开销远低于全局临界区 |
| `parallel for` | 自动划分迭代空间，自动私有化循环变量 |
| 规范形式 | 循环次数可预先确定，循环变量不得在体内被修改 |

### 一个需要强调的结论

> 满足规范形式，只说明编译器**能够**分担这个循环；迭代之间是否真的独立，编译器不作检查，在一般情形下（存在指针别名与间接寻址时）也无法自动判定。

这句话正是下一个实验的起点。实验三将展示两个满足规范形式、编译无任何告警、却给出错误结果的循环，并系统讨论如何识别与消除循环携带依赖。